# POPSIGN — Landmark Extraction Driver (staged, one dataset at a time)

**What this notebook does.** Drives MediaPipe Holistic landmark extraction over
the raw POPSIGN videos using `sb.extract.holistic`. **Pipeline stage 0 for
POPSIGN** — everything downstream (feature caches, training) consumes its npz
output. For a cheap single-video/small-batch sanity check instead, see
`popsign.0.dataset.pilot.ipynb` (split out 2026-09-13).

## Staged design (2026-09-13 rewrite)

**Never more than one dataset's raw video sits on disk at a time.** Each of the
3 enabled POPSIGN train parts (~170-200GB each) and the test set (~17GB) is
downloaded fully, extracted fully, verified, and then its kagglehub cache is
**deleted** before the next stage starts — no more `resolve_datasets()` call
that pulls every enabled dataset onto disk together. All downloads go through
kagglehub's own default cache (`~/.cache/kagglehub/datasets/...`) — never an
external drive (`D:`/`E:` overrides were dropped 2026-09-13; they broke
resumable re-download checks and needed manual disk management this now does
for you).

Stage order (the cells below are meant to be run **top to bottom, in this
order** — each stage's cache-delete cell is guarded by an assertion that its
extraction actually finished, so running out of order just fails loudly
instead of silently deleting unextracted video):

1. Train part 1/3 (`train-a-e`) — download, manifest, worker-count pilot
   (picks the operating point used by every later stage), extract, delete.
2. Train part 2/3 (`train-f-m`) — download, manifest, extract, delete.
3. Train part 3/3 (`train-n-s`) — download, manifest, extract, delete.
4. Test — download once, manifest, extract in **4 quarters** (a checkpoint
   cadence, not 4 separate downloads — see "Why test isn't quartered by
   download" below), delete.
5. QC over both splits.

## Execution model — this notebook does not run the worker pool

**Extraction itself runs as a script, not in a cell.** `multiprocessing` uses
the *spawn* start method on Windows: spawned children re-import `__main__`,
which inside a Jupyter kernel is the kernel rather than a module, so the
workers never start and the pool blocks forever. `extraction._assert_pool_usable`
refuses that case up front, so the cells below would raise rather than hang.
Every per-stage **extract** cell is a **handoff cell**: it prints the exact
`sb-extract` command to run in a terminal, then reads progress back off the
manifest on disk — re-run it any time to check progress. (The pilot notebook's
single-video check is the one exception: `n_workers=1` runs in-process.)

## Why test isn't quartered by *download*

Unlike train (already split into 4 separate Kaggle datasets), POPSIGN test is
**one** Kaggle dataset — there is no bulk "download a quarter of it" API.
kagglehub's `path=<file>` download fetches exactly one file with no bulk mode,
and per-file downloads measured **~5s/file** on this connection (the pilot
notebook's finding) — a real quarter (~8,400 files) would take ~12h of
download overhead alone, far worse than one continuous ~17GB archive transfer.
So test is downloaded **once**, and the 4 quarters below are an *extraction*
checkpoint cadence (`--limit`-based, resumable) sharing that one download — its
cache is deleted only after all 4 quarters finish, matching "when one dataset
is extracted, delete its cache."

## Output

One npz per video at `<root>/data/raw/popsign/<split>/<label>/<id>.npz`, where
`<root>` is `POPSIGN_LANDMARKS_DRIVE` from `.env` when set (else the gitignored
`src/data/raw/` tree). Extracted landmarks are **persistent** data; only the
downloaded source video (kagglehub cache) and the worker-count pilot's npz are
temporary.

| key | content |
|---|---|
| `landmarks` | `(T, 543, 3)` float16, **GISLR holistic row order** (face 0-467, left_hand 468-488, pose 489-521, right_hand 522-542), NaN where undetected — `sb.core.subsets` indices apply unchanged |
| `fps` / `num_frames` | video metadata |

## Artifacts of this notebook (under `data/cache/popsign/`)

| path | content |
|---|---|
| `dataframes/train_parts/<name>.csv` | one manifest slice per downloaded train part (label, id, file_path) |
| `dataframes/train.csv` | cumulative concat of every `train_parts/*.csv` downloaded so far — what `sb-extract run train` reads |
| `dataframes/test.csv` | the test manifest (single dataset, no per-part split) |
| `extraction/pilot_sample.json`, `pilot_results.csv`, `pilot_throughput.png`, `eta.json` | worker-count pilot (run once, against train part 1) |
| `extraction/{train,test}_summary.json` | the CLI's summary for each `run` invocation |
| `<out_dir>/<split>/_manifest.json` | per-video extraction state (lives next to the npz output) |

## Resumability

The extraction module follows the repo manifest pattern (TODO §1.2): npz
written atomically (temp + `os.replace`) **before** the video is marked
`done`; `done` skipped, `failed` retried on re-run; manifest saves are atomic
and batched. Because state lives in the manifest and not in the kernel, Ctrl-C
on the CLI is safe and re-running the same command resumes. A stage's
cache-delete cell only ever removes the **downloaded video cache**, never the
extraction manifest or npz output — re-downloading a part after its cache was
cleared and re-running its extract cell is a no-op (everything is already
`done`).

## Resource cap (the ≤70% rule)

- **CPU**: default worker count = `floor(0.70 × logical cores)`; each worker is
  pinned to single-threaded math libs. The pilot sweep verifies measured CPU%.
- **RAM**: the job feeder blocks while system RAM ≥ 70% (psutil backpressure).
- **GPU**: untouched — the MediaPipe GPU delegate is Ubuntu-only, so extraction
  is CPU-only by construction.

## Design decisions

- Manifests are built by **walking each part's freshly-downloaded tree**
  (`<root>/<sign>/<sign>/<id>.mp4`) right after its download, not by listing
  files via the Kaggle API — the actual bytes have to be fetched anyway (bulk
  archive download, the efficient path — see "Why test isn't quartered"
  above), so walking the extracted tree is free and exact.
- `train.csv` is rebuilt as the concat of every `train_parts/*.csv` written so
  far, every time a new part's manifest is added — `sb-extract run train`
  reads that one path unchanged, so **no CLI changes were needed** for the
  staged redesign; already-`done` rows for a part whose cache has since been
  cleared are simply skipped (`pending_jobs` only checks the npz artifact).
- Test's 4 quarters reuse the same `--limit`-based resumability the CLI
  already had (`run test --limit N` four times just processes the next batch
  of pending videos each time) — no new CLI flag for "quarter index" needed.
- Only 3 of 4 POPSIGN train parts are enabled (`sb.core.paths.ENABLED_TRAIN_INDICES`)
  — `train-t-z` stays disabled; add a 4th stage below (copy part 3's cells,
  bump the index) when it's enabled.

In [ ]:
# ============================================================
# Imports, config, and shared per-stage helper functions
# ============================================================
import json
import math
from multiprocessing import cpu_count
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import sb.extract.holistic as ex
import sb.core.paths as paths
from sb.core.paths import CACHE_DIR, DATASET_IDS, ROOT_DIR, TEMP_DIR


SEED = 42  # pilot sampling seed (must match extract_popsign.py)
DATAFRAMES_DIR = CACHE_DIR / "popsign" / "dataframes"
TRAIN_PARTS_DIR = DATAFRAMES_DIR / "train_parts"  # one manifest slice per downloaded train part
TRAIN_CSV = DATAFRAMES_DIR / "train.csv"          # cumulative concat of TRAIN_PARTS_DIR/*.csv
TEST_CSV = DATAFRAMES_DIR / "test.csv"
TRAIN_PARTS_DIR.mkdir(parents=True, exist_ok=True)
NB_CACHE_DIR = CACHE_DIR / "popsign" / "extraction"
NB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

PART_NAMES = ["train-a-e", "train-f-m", "train-n-s", "train-t-z"]  # index-aligned with DATASET_IDS["TRAIN"]

PILOT_MAX_VIDEOS = 100  # hard cap on the worker-count pilot batch
WORKER_COUNTS = (6, 10, 14, 19)  # sweep; 19 = floor(0.70 × 28 logical cores)
VIDEOS_PER_TRIAL = 20  # per worker count (4 × 20 = 80 ≤ 100)
CPU_FRACTION = 0.70  # resource cap
RAM_LIMIT_PCT = 70.0

TEST_N_QUARTERS = 4  # extraction checkpoint cadence for the test stage (see title cell)

# Detector thresholds for the bulk run — an arm of the confidence sweep
# (confidence_tuning/configs.json), so the output records what produced it.
# Interim choice per docs/reports/confidence-tuning.md; None = MediaPipe defaults.
CONFIDENCE_CONFIG = "default"

OUT_ROOT = ex.landmarks_root()  # resolved from POPSIGN_LANDMARKS_DRIVE / fallback

# ---- CLI handoff -------------------------------------------------------
REPO_ROOT = ROOT_DIR
EXTRACT_CLI = REPO_ROOT / ".venv" / "Scripts" / "sb-extract.exe"


def extract_command(*args: str) -> str:
    """The exact shell command for an `sb-extract` invocation."""
    parts = [EXTRACT_CLI.relative_to(REPO_ROOT).as_posix(), *args]
    if CONFIDENCE_CONFIG:
        parts += ["--confidence", CONFIDENCE_CONFIG]
    parts += ["--cpu-fraction", str(CPU_FRACTION), "--ram-limit-pct", str(RAM_LIMIT_PCT)]
    return " ".join(parts)


def walk_manifest(root: Path) -> pd.DataFrame:
    """<root>/<sign>/<sign>/<id>.mp4 -> a (file_path, label, id) dataframe for
    one downloaded part/split, with the same id-uniqueness and label/filename
    agreement checks the old single combined-manifest cell used to run."""
    rows = [{"file_path": p.as_posix(), "label": p.parts[-2], "id": p.stem}
            for p in root.rglob("*.mp4")]
    df = pd.DataFrame(rows).sort_values("id", ignore_index=True)
    assert len(df), f"no .mp4 files found under {root}"
    assert df["id"].is_unique, f"{root}: duplicate video ids"
    bad = df[~df.apply(lambda r: f"-{r.label}-" in r.id, axis=1)]
    assert bad.empty, (f"{root}: {len(bad)} ids disagree with their folder label, "
                       f"e.g. {bad.head(3)['id'].tolist()}")
    return df


def save_part_manifest(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Persist one train part's manifest slice, then rebuild the cumulative
    train.csv (concat of every part manifest saved so far) — the one path
    `sb-extract run train` reads, unchanged by the staged redesign."""
    df.to_csv(TRAIN_PARTS_DIR / f"{name}.csv", index=False)
    parts = sorted(TRAIN_PARTS_DIR.glob("*.csv"))
    combined = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
    assert combined["id"].is_unique, "duplicate video ids across train parts"
    combined.to_csv(TRAIN_CSV, index=False)
    print(f"  {name}: {len(df):,} videos · {df['label'].nunique()} labels -> {TRAIN_CSV.name} "
          f"now {len(combined):,} videos total ({len(parts)} part(s))")
    return combined


def part_progress(ids, split: str) -> dict:
    """Extraction status restricted to ONE stage's ids — unlike split_progress
    (the whole split's manifest), this is what gates a stage's cache-delete
    cell: a part/quarter must be fully done before its videos disappear."""
    manifest = ex.load_manifest(OUT_ROOT / split)
    statuses = pd.Series({i: manifest.get(i, {}).get("status", "pending") for i in ids})
    done = int((statuses == "done").sum())
    return {"total": len(ids), "done": done,
           "failed": int((statuses == "failed").sum()),
           "pct": round(100 * done / len(ids), 1) if len(ids) else None}


def split_progress(split: str) -> dict:
    """Extraction state for the WHOLE split, read off the manifest on disk —
    grows as more parts/quarters are added; see part_progress for one stage."""
    csv = TRAIN_CSV if split == "train" else TEST_CSV
    total = len(pd.read_csv(csv)) if csv.exists() else None
    statuses = pd.Series(
        {k: v["status"] for k, v in ex.load_manifest(OUT_ROOT / split).items()}, dtype=object)
    done = int((statuses == "done").sum()) if len(statuses) else 0
    return {"split": split, "total": total if total is not None else "no manifest csv",
           "done": done, "failed": int((statuses == "failed").sum()) if len(statuses) else 0,
           "pct": round(100 * done / total, 1) if total else None}


print(f"logical cores : {cpu_count()}  → default workers {ex.default_n_workers(CPU_FRACTION)}")
print(f"output root   : {OUT_ROOT.resolve()}")
print(f"model         : {ex.DEFAULT_MODEL_PATH}  (exists: {ex.DEFAULT_MODEL_PATH.exists()})")
print(f"nb cache      : {NB_CACHE_DIR}")
print(f"confidence    : {CONFIDENCE_CONFIG or 'MediaPipe defaults'}")
print(f"extract CLI   : {EXTRACT_CLI}  (exists: {EXTRACT_CLI.exists()})")
print(f"enabled train : {[PART_NAMES[i] for i in paths.ENABLED_TRAIN_INDICES]}")

## 1. Train part 1/3 — `train-a-e`: download + manifest

Downloads the whole part (~170GB, one continuous archive transfer — the
efficient path) to kagglehub's default cache, then walks its tree into a
manifest slice. Safe to re-run: kagglehub skips a part already fully cached.

In [ ]:
# ============================================================
# STAGE: train part 1/3 (train-a-e) — download + manifest
# ============================================================
PART_INDEX = 0
PART_NAME = PART_NAMES[PART_INDEX]
PART_HANDLE = DATASET_IDS["TRAIN"][PART_INDEX]

part_root = paths.train_dir(PART_INDEX)
print(f"{PART_NAME}: downloaded to {part_root}")
part_df = walk_manifest(part_root)
train_df = save_part_manifest(part_df, PART_NAME)

## 2. Worker-count pilot — parameter optimization (run once, against part 1)

**Handoff cell.** The benchmark runs as `extract_popsign.py pilot`; the cell
below prints the command and, once `pilot_results.csv` exists, plots it. Needs
part 1's videos already on disk (previous cell) to have something to time.
The chosen worker count (`eta.json`) is reused by every later stage — this
section only needs to run once for the whole notebook.

The script sweeps `WORKER_COUNTS` on **disjoint** slices of a seeded 100-video
sample from `train.csv` (at this point, just part 1). Pilot npz files are
throwaway (`data/temp/popsign_pilot/w<N>/`), deleted by the cleanup cell below.

In [ ]:
# ============================================================
# Pilot sweep — HANDOFF: run the benchmark in a terminal, then plot it here
# ============================================================
print("Run this in a terminal (worker pools cannot start from a Jupyter kernel):\n")
print("   ", extract_command("pilot", "--worker-counts", *map(str, WORKER_COUNTS),
                             "--videos-per-trial", str(VIDEOS_PER_TRIAL)))
print(f"\n~{len(WORKER_COUNTS) * VIDEOS_PER_TRIAL} extractions; writes "
      f"{NB_CACHE_DIR / 'pilot_results.csv'}\nThen re-run this cell to plot it.\n")

_results_csv = NB_CACHE_DIR / "pilot_results.csv"
if not _results_csv.exists():
    print("no pilot_results.csv yet — run the command above first")
else:
    pilot_results = pd.read_csv(_results_csv)
    fig, (ax_t, ax_c) = plt.subplots(2, 1, figsize=(6.4, 5), sharex=True,
                                     gridspec_kw={"height_ratios": [3, 2]})
    ax_t.plot(pilot_results["n_workers"], pilot_results["videos_per_s"],
              "o-", color="#2f6fd0", lw=2, ms=8)
    ax_t.set_ylabel("videos / s")
    ax_t.set_title("Pilot: extraction throughput vs worker count", loc="left")
    _best = pilot_results.loc[pilot_results["videos_per_s"].idxmax()]
    ax_t.annotate(f"{_best['videos_per_s']:.2f} v/s @ {int(_best['n_workers'])}w",
                  xy=(_best["n_workers"], _best["videos_per_s"]),
                  xytext=(0, 10), textcoords="offset points",
                  ha="center", fontsize=9, color="#2f6fd0", fontweight="bold")
    ax_c.plot(pilot_results["n_workers"], pilot_results["cpu_mean_pct"],
              "s-", color="#d1651a", lw=2, ms=7)
    ax_c.axhline(CPU_FRACTION * 100, color="#7a8794", ls="--", lw=1.2)
    ax_c.annotate(f"cap {CPU_FRACTION:.0%}", xy=(0.99, CPU_FRACTION * 100),
                  xycoords=("axes fraction", "data"), xytext=(0, 4),
                  textcoords="offset points", ha="right", fontsize=8.5, color="#7a8794")
    ax_c.set_ylabel("mean CPU %")
    ax_c.set_xlabel("worker processes")
    ax_c.set_ylim(0, 105)
    for ax in (ax_t, ax_c):
        ax.grid(alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(NB_CACHE_DIR / "pilot_throughput.png", dpi=130)
    plt.show()
    display(pilot_results)

In [ ]:
# ============================================================
# Pick the operating point -> eta.json (reused by every later stage)
# ============================================================
pilot_results = pd.read_csv(NB_CACHE_DIR / "pilot_results.csv")
_ok = pilot_results[pilot_results["cpu_mean_pct"] <= CPU_FRACTION * 100 + 5]
best = (_ok if len(_ok) else pilot_results).sort_values("videos_per_s", ascending=False).iloc[0]
eta = {"best_n_workers": int(best["n_workers"]), "videos_per_s": float(best["videos_per_s"])}
(NB_CACHE_DIR / "eta.json").write_text(json.dumps(eta, indent=2))
print(json.dumps(eta, indent=2))

In [ ]:
# ============================================================
# Pilot cleanup — the sweep's npz output is throwaway (data/temp policy)
# ============================================================
assert (NB_CACHE_DIR / "eta.json").exists(), \
    "run the ETA cell first — cleanup would discard the unmeasured pilot output"
paths.cleanup_temp()
print(f"deleted temp tree ({TEMP_DIR}) — pilot measurements persist in {NB_CACHE_DIR}")

## 3. Train part 1/3 — `train-a-e`: extract + verify + delete cache

**Handoff cell**, then a guarded delete: the cache-delete cell only runs once
every video in this part is confirmed `done`.

In [ ]:
# ============================================================
# Train part 1/3 — extract: HANDOFF + progress readback
# ============================================================
PART_INDEX = 0

print("Run this in a terminal:\n")
print("   ", extract_command("run", "train"))
print("\nProgress for this part specifically (re-run any time):")
print("   ", part_progress(part_df["id"].tolist(), "train"))
print("\nProgress for the whole train split so far:")
print("   ", split_progress("train"))

In [ ]:
# ============================================================
# Train part 1/3 — delete cache (guarded: only once fully extracted)
# ============================================================
_prog = part_progress(part_df["id"].tolist(), "train")
assert _prog["done"] == _prog["total"], f"train-a-e not fully extracted yet: {_prog}"
paths.clear_dataset_cache(DATASET_IDS["TRAIN"][0])
print(f"cleared cache for {DATASET_IDS['TRAIN'][0]} ({_prog['done']}/{_prog['total']} done)")

## 4. Train part 2/3 — `train-f-m`: download + manifest + extract + delete

Same three steps as part 1, minus the pilot (its `eta.json` is already picked
and reused here).

In [ ]:
# ============================================================
# STAGE: train part 2/3 (train-f-m) — download + manifest
# ============================================================
PART_INDEX = 1
PART_NAME = PART_NAMES[PART_INDEX]
PART_HANDLE = DATASET_IDS["TRAIN"][PART_INDEX]

part_root = paths.train_dir(PART_INDEX)
print(f"{PART_NAME}: downloaded to {part_root}")
part_df = walk_manifest(part_root)
train_df = save_part_manifest(part_df, PART_NAME)

In [ ]:
# ============================================================
# Train part 2/3 — extract: HANDOFF + progress readback
# ============================================================
print("Run this in a terminal:\n")
print("   ", extract_command("run", "train"))
print("\nProgress for this part specifically (re-run any time):")
print("   ", part_progress(part_df["id"].tolist(), "train"))
print("\nProgress for the whole train split so far:")
print("   ", split_progress("train"))

In [ ]:
# ============================================================
# Train part 2/3 — delete cache (guarded: only once fully extracted)
# ============================================================
_prog = part_progress(part_df["id"].tolist(), "train")
assert _prog["done"] == _prog["total"], f"train-f-m not fully extracted yet: {_prog}"
paths.clear_dataset_cache(DATASET_IDS["TRAIN"][1])
print(f"cleared cache for {DATASET_IDS['TRAIN'][1]} ({_prog['done']}/{_prog['total']} done)")

## 5. Train part 3/3 — `train-n-s`: download + manifest + extract + delete

In [ ]:
# ============================================================
# STAGE: train part 3/3 (train-n-s) — download + manifest
# ============================================================
PART_INDEX = 2
PART_NAME = PART_NAMES[PART_INDEX]
PART_HANDLE = DATASET_IDS["TRAIN"][PART_INDEX]

part_root = paths.train_dir(PART_INDEX)
print(f"{PART_NAME}: downloaded to {part_root}")
part_df = walk_manifest(part_root)
train_df = save_part_manifest(part_df, PART_NAME)

In [ ]:
# ============================================================
# Train part 3/3 — extract: HANDOFF + progress readback
# ============================================================
print("Run this in a terminal:\n")
print("   ", extract_command("run", "train"))
print("\nProgress for this part specifically (re-run any time):")
print("   ", part_progress(part_df["id"].tolist(), "train"))
print("\nProgress for the whole train split so far:")
print("   ", split_progress("train"))

In [ ]:
# ============================================================
# Train part 3/3 — delete cache (guarded: only once fully extracted)
# ============================================================
_prog = part_progress(part_df["id"].tolist(), "train")
assert _prog["done"] == _prog["total"], f"train-n-s not fully extracted yet: {_prog}"
paths.clear_dataset_cache(DATASET_IDS["TRAIN"][2])
print(f"cleared cache for {DATASET_IDS['TRAIN'][2]} ({_prog['done']}/{_prog['total']} done)")

## 6. Test — download once + manifest

Downloaded once (~17GB, one archive transfer) — see the title cell for why
this isn't split into 4 separate downloads.

In [ ]:
# ============================================================
# STAGE: test — download + manifest
# ============================================================
test_root = paths.test_dir()
print(f"test: downloaded to {test_root}")
test_df = walk_manifest(test_root)
test_df.to_csv(TEST_CSV, index=False)
print(f"  test: {len(test_df):,} videos · {test_df['label'].nunique()} labels -> {TEST_CSV}")

TEST_QUARTER_SIZE = math.ceil(len(test_df) / TEST_N_QUARTERS)
print(f"  quarter size: {TEST_QUARTER_SIZE} videos ({TEST_N_QUARTERS} quarters)")

## 7. Test — extract in 4 quarters, then delete cache

Each quarter is the **same** `run test --limit <quarter size>` command —
`--limit` only ever processes *pending* videos, so four sequential runs
naturally cover the set in four checkpoints without any new CLI flag. Run
quarter *n* only after quarter *n-1* shows `pct: 100.0` for its slice (or just
keep re-running the same command — a finished quarter's videos are already
`done` and get skipped instantly).

In [ ]:
# ============================================================
# Test — quarter 1/4: HANDOFF + progress readback
# ============================================================
print("Run this in a terminal:\n")
print("   ", extract_command("run", "test", "--limit", str(TEST_QUARTER_SIZE)))
print("\nProgress for the whole test split (re-run any time):")
print("   ", split_progress("test"))

In [ ]:
# ============================================================
# Test — quarter 2/4: HANDOFF + progress readback
# ============================================================
print("Run this in a terminal:\n")
print("   ", extract_command("run", "test", "--limit", str(TEST_QUARTER_SIZE)))
print("\nProgress for the whole test split (re-run any time):")
print("   ", split_progress("test"))

In [ ]:
# ============================================================
# Test — quarter 3/4: HANDOFF + progress readback
# ============================================================
print("Run this in a terminal:\n")
print("   ", extract_command("run", "test", "--limit", str(TEST_QUARTER_SIZE)))
print("\nProgress for the whole test split (re-run any time):")
print("   ", split_progress("test"))

In [ ]:
# ============================================================
# Test — quarter 4/4 (final): HANDOFF + progress readback
# ============================================================
print("Run this in a terminal:\n")
print("   ", extract_command("run", "test", "--limit", str(TEST_QUARTER_SIZE)))
print("\nProgress for the whole test split (re-run any time):")
print("   ", split_progress("test"))

In [ ]:
# ============================================================
# Test — delete cache (guarded: only once ALL 4 quarters are fully extracted)
# ============================================================
_prog = split_progress("test")
assert _prog["pct"] == 100.0, f"test not fully extracted yet: {_prog}"
paths.clear_dataset_cache(DATASET_IDS["TEST"])
print(f"cleared cache for {DATASET_IDS['TEST']} ({_prog['done']}/{_prog['total']} done)")

## 8. QC — manifest stats, failures, npz spot-checks

Reads only the split manifests and npz artifacts (no video access, no
downloaded cache needed) — cheap to re-run at any point mid-extraction.

In [ ]:
# ============================================================
# QC over both splits — progress, failures, npz spot-check
# ============================================================
N_QC_SAMPLES = 5   # npz files load-checked per split
rng = np.random.default_rng(SEED)

for split in ["train", "test"]:
    manifest = ex.load_manifest(OUT_ROOT / split)
    if not manifest:
        print(f"{split}: no manifest yet — extraction not started\n")
        continue
    st = pd.Series({k: v["status"] for k, v in manifest.items()})
    done = st[st == "done"].index
    print(f"{split}: {len(st):,} attempted — {(st == 'done').sum():,} done · "
          f"{(st == 'failed').sum():,} failed")
    fails = {k: v["error"] for k, v in manifest.items() if v["status"] == "failed"}
    if fails:
        print("  failure reasons (top):")
        print(pd.Series(list(fails.values())).value_counts().head(5).to_string())
    nf = pd.Series({k: manifest[k].get("n_frames") for k in done}).dropna()
    if len(nf):
        print(f"  frames/video: median {nf.median():.0f} · p5 {nf.quantile(.05):.0f} "
              f"· p95 {nf.quantile(.95):.0f} · zero-frame videos {(nf == 0).sum()}")
    for vid in rng.choice(done, size=min(N_QC_SAMPLES, len(done)), replace=False):
        z = np.load(manifest[vid]["artifact"])
        lm = z["landmarks"]
        assert lm.ndim == 3 and lm.shape[1:] == (543, 3), f"bad shape {lm.shape}: {vid}"
        nan_frac = float(np.isnan(lm.astype(np.float32)).mean()) if len(lm) else 1.0
        print(f"  ok {vid}: {lm.shape} float16 · NaN {nan_frac:.1%} · fps {float(z['fps']):.0f}")
    _sz = sum(Path(manifest[k]["artifact"]).stat().st_size
              for k in done if Path(manifest[k]["artifact"]).exists())
    print(f"  on disk: {_sz / 1e9:.2f} GB\n")

## Caveats & follow-ups

- **The worker pool cannot run in this notebook.** `multiprocessing` spawn +
  Jupyter on Windows hangs silently at 0% forever. The pilot notebook's
  `n_workers=1` in-process mode is the only in-notebook extraction; everything
  else here is a handoff cell running `sb-extract` in a terminal.
- **A stage's cache-delete cell only deletes the downloaded video cache** —
  never the extraction manifest or npz output (`data/raw/popsign/...`), and
  it's guarded by an assertion that the stage actually finished. Running cells
  out of order fails loudly (`AssertionError: ... not fully extracted yet`)
  rather than silently deleting unextracted video.
- **`train-t-z` (index 3) is still disabled** (`sb.core.paths.ENABLED_TRAIN_INDICES`).
  Enabling it means adding a 4th train stage here (copy part 3's three cells,
  bump `PART_INDEX` to 3) — TODO §2.2.
- **`CONFIDENCE_CONFIG` is an interim choice.** The confidence sweep
  (`docs/reports/confidence-tuning.md`) has 2 of 7 arms measured; `"default"`
  leads the paired comparison that exists so far.
- `fps` is taken from the container metadata (`cv2.CAP_PROP_FPS`, fallback 30
  when the container reports 0) and is stored per npz; downstream consumers
  should not assume a uniform frame rate across POPSIGN.
- Landmark values are float16 (halves storage; x/y are normalized [0,1] where
  ~3 significant digits survive, which is far above landmark jitter).